# Import data

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go


## Load the CSV file into a DataFrame

The file has headers in the first row. `pandas.read_csv()` reads them automatically and creates a `DataFrame` from the data.

In [4]:


csv_candidates = (
    Path.cwd() / "src" / "metal_bands.csv",
    Path.cwd().parent / "src" / "metal_bands.csv",
)
csv_path = next((path for path in csv_candidates if path.is_file()), None)

if csv_path is None:
    raise FileNotFoundError("Could not find src/metal_bands.csv.")

df = pd.read_csv(csv_path, dtype={"Band ID": "string"})

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
print("Headers:", df.columns.tolist())
display(df.head())

Loaded 183397 rows and 7 columns.
Headers: ['Band ID', 'Name', 'URL', 'Country', 'Genre', 'Status', 'Photo_URL']


,Band ID,Name,URL,Country,Genre,Status,Photo_URL
0,3540442600,A // Solution,https://www.metal-archives.com/bands/A_--_Solu...,United States,Crust Punk/Thrash Metal,Split-up,NaN
1,3540525193,A Black Cold Diamond,https://www.metal-archives.com/bands/A_Black_C...,Colombia,Heavy/Doom Metal,Active,https://www.metal-archives.com/images/3/5/4/0/...
2,3540473101,A Billion Limbs,https://www.metal-archives.com/bands/A_Billion...,United Kingdom,Groove Metal/Deathcore,Active,NaN
3,3540535978,A Band Named Jon,https://www.metal-archives.com/bands/A_Band_Na...,United States,Brutal Death Metal/Grindcore,Active,https://www.metal-archives.com/images/3/5/4/0/...
4,3540352307,A Balance of Power,https://www.metal-archives.com/bands/A_Balance...,United States,Melodic Death Metal/Metalcore,Active,https://www.metal-archives.com/images/3/5/4/0/...


#ADDING COUNTER

In [5]:
df["COUNTER"]=1


#HOW MANY BANDS PER COUNTRY 

In [6]:
COUNTRY_BANDS=df[["Country","COUNTER"]].groupby("Country").sum().sort_values(by="COUNTER",ascending=False)

#RESULT

#ONLY 100+

In [7]:
COUNTRY_BANDS_100=COUNTRY_BANDS[COUNTRY_BANDS["COUNTER"] > 100]

In [8]:
print(COUNTRY_BANDS_100)

                COUNTER
Country                
United States     41386
Germany           13840
Brazil             8408
Italy              8100
United Kingdom     7513
...                 ...
Taiwan              119
Bangladesh          111
Panama              108
Malta               104
Luxembourg          102

[74 rows x 1 columns]


In [ ]:
country_centroids = {
    "United States": (39.8, -98.6), "Germany": (51.2, 10.4),
    "Brazil": (-10.8, -52.9), "Italy": (42.8, 12.6),
    "United Kingdom": (54.5, -2.5), "France": (46.2, 2.2),
    "Canada": (56.1, -106.3), "Russia": (61.5, 105.3),
    "Sweden": (60.1, 18.6), "Finland": (64.0, 26.0),
    "Spain": (40.2, -3.7), "Poland": (52.1, 19.1),
    "Mexico": (23.6, -102.5), "Australia": (-25.3, 133.8),
    "Argentina": (-34.0, -64.0), "Chile": (-30.0, -71.0),
    "Netherlands": (52.1, 5.3), "Japan": (36.2, 138.0),
    "Greece": (39.0, 22.0), "Indonesia": (-2.0, 118.0),
    "Norway": (61.0, 8.0), "Colombia": (4.0, -72.0),
    "Czechia": (49.8, 15.5), "Portugal": (39.6, -8.0),
    "Belgium": (50.5, 4.5), "Austria": (47.5, 14.5),
    "Ukraine": (49.0, 32.0), "Hungary": (47.2, 19.5),
    "Switzerland": (46.8, 8.2), "Denmark": (56.0, 9.5),
    "Malaysia": (4.0, 102.0), "Ecuador": (-1.4, -78.4),
    "Slovakia": (48.7, 19.5), "Peru": (-9.2, -75.0),
    "T\u00fcrkiye": (39.0, 35.0), "Croatia": (45.1, 15.2),
    "New Zealand": (-41.0, 174.0), "Venezuela": (7.0, -66.0),
    "Romania": (45.9, 24.9), "Ireland": (53.4, -8.0),
    "Serbia": (44.0, 20.8), "Belarus": (53.5, 28.0),
    "Bulgaria": (42.7, 25.5), "China": (35.9, 103.8),
    "Philippines": (12.9, 122.8), "Costa Rica": (9.7, -84.0),
    "Bolivia": (-16.3, -64.7), "India": (22.5, 79.0),
    "Israel": (31.5, 34.8), "Singapore": (1.35, 103.8),
    "Slovenia": (46.1, 14.8), "South Africa": (-29.0, 24.0),
    "Korea, South": (36.5, 128.0), "Estonia": (58.7, 25.0),
    "Thailand": (15.0, 101.0), "Uruguay": (-32.8, -56.0),
    "Paraguay": (-23.4, -58.4), "Lithuania": (55.2, 23.9),
    "Iceland": (64.9, -18.6), "Puerto Rico": (18.2, -66.5),
    "El Salvador": (13.8, -88.9), "Guatemala": (15.8, -90.2),
    "Bosnia and Herzegovina": (44.2, 17.7), "Latvia": (56.9, 24.6),
    "Iran": (32.4, 53.7), "Cuba": (21.5, -78.0),
    "North Macedonia": (41.6, 21.7), "Taiwan": (23.7, 121.0),
    "Bangladesh": (24.0, 90.0), "Panama": (8.5, -80.0),
    "Malta": (35.9, 14.4), "Luxembourg": (49.8, 6.1),
}

map_data = COUNTRY_BANDS_100.reset_index().rename(columns={"Country": "country"})
unmapped_countries = sorted(set(map_data["country"]) - set(country_centroids))
if unmapped_countries:
    print("Countries without geographic centroids (omitted):", ", ".join(unmapped_countries))
map_data = map_data[map_data["country"].isin(country_centroids)].copy()
map_data[["latitude", "longitude"]] = map_data["country"].map(country_centroids).apply(pd.Series)

counts = map_data["COUNTER"].astype(float)
sqrt_counts = np.sqrt(counts)
count_span = sqrt_counts.max() - sqrt_counts.min()
skull_sizes = 14 + 34 * (sqrt_counts - sqrt_counts.min()) / count_span if count_span else 24
skull = "\U0001F480"

fig = go.Figure(go.Scattergeo(
    lat=map_data["latitude"],
    lon=map_data["longitude"],
    mode="text",
    text=[skull] * len(map_data),
    textfont=dict(size=skull_sizes, family="Segoe UI Emoji, Noto Color Emoji, sans-serif"),
    customdata=map_data[["country", "COUNTER"]].to_numpy(),
    hovertemplate="<b>%{customdata[0]}</b><br>Bands: %{customdata[1]}<extra></extra>",
    textposition="middle center",
    showlegend=False,
))

fig.update_layout(
    title="Metal bands by country (larger skull = more bands)",
    template="plotly_dark",
    paper_bgcolor="rgb(13, 20, 30)",
    plot_bgcolor="rgb(13, 20, 30)",
    height=650,
    geo=dict(
        scope="world",
        projection_type="natural earth",
        showframe=False,
        showland=True,
        landcolor="rgb(46, 58, 70)",
        showocean=True,
        oceancolor="rgb(13, 20, 30)",
        showcountries=True,
        countrycolor="rgb(105, 125, 145)",
    ),
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()
